# Parameter Comparison: hydro-param vs NHM Reference

Compare pywatershed parameters generated by hydro-param against the NHM reference
dataset (`drb_2yr/parameters_dis_both.nc`). Both cover the Delaware River Basin domain
(765 HRUs, 456 segments).

**Reference:** NHM-PRMS calibrated parameters distributed with pywatershed.

**Expected differences:**
- Different source data vintages (NHM used older NLCD/soils/DEM)
- Different zonal stats engines (exactextract vs legacy PRMS GIS)
- Circular vs arithmetic aspect mean
- NHD spatial-join mismatches for segment-level cumulative area and slope

**Known limitations (seg_\* parameters):**
- `seg_cum_area`: ~28% of segments have large errors from NHD spatial-join mismatches
- `seg_slope`: same root cause — wrong NHD flowline matched to segment
- `segment_type`: missing lake classification (needs waterbody overlay)
- `seg_depth`: constant 1.0 default vs empirically-derived reference values

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Paths — adjust if running from a different working directory
REF_PARAM = Path("../data/pywatershed_gis/drb_2yr/parameters_dis_both.nc")
HP_PARAM = Path("../pw-check/models/pywatershed/parameters.nc")
FABRIC_PATH = Path("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
SEGMENT_PATH = Path("../data/pywatershed_gis/drb_2yr/nsegment.gpkg")

for p, label in [
    (REF_PARAM, "Reference"),
    (HP_PARAM, "hydro-param"),
    (FABRIC_PATH, "HRU fabric"),
]:
    assert p.exists(), f"{label} not found: {p}"
    print(f"{label}: {p}")


## Load Parameters

In [ ]:
# Both are plain NetCDF — load with xarray directly
ref_ds = xr.open_dataset(REF_PARAM)
hp_ds = xr.open_dataset(HP_PARAM)

print(f"Reference:  {dict(ref_ds.sizes)}, {len(ref_ds.data_vars)} variables")
print(f"hydro-param: {dict(hp_ds.sizes)}, {len(hp_ds.data_vars)} variables")

## Coordinate Alignment

Reference uses `nhm_id` coordinate on the `nhru` dimension (5307–7251).
hydro-param uses `nhru` coordinate values offset by +1 (5308–7252).
Both have 765 HRUs over the same DRB domain. We align by position (same spatial ordering).

For segments, both use `nhm_seg` / `nsegment` with matching values.

In [ ]:
# HRU alignment
ref_hru_ids = ref_ds.nhm_id.values
hp_hru_ids = hp_ds.nhru.values
offset = hp_hru_ids - ref_hru_ids
assert np.all(offset == offset[0]), "HRU ID offset is not constant — cannot align by position"
print(f"HRU IDs: ref {ref_hru_ids.min()}–{ref_hru_ids.max()}, "
      f"hp {hp_hru_ids.min()}–{hp_hru_ids.max()} (offset={offset[0]:+d})")
print(f"Both have {len(ref_hru_ids)} HRUs — aligning by position.")

# Segment alignment
ref_seg_ids = ref_ds.nhm_seg.values
hp_seg_ids = hp_ds.nsegment.values
seg_match = np.array_equal(ref_seg_ids, hp_seg_ids)
print(f"\nSegment IDs match: {seg_match} ({len(ref_seg_ids)} segments)")

## Variable Inventory

In [ ]:
ref_vars = set(ref_ds.data_vars)
hp_vars = set(hp_ds.data_vars)
common = sorted(ref_vars & hp_vars)
only_ref = sorted(ref_vars - hp_vars)
only_hp = sorted(hp_vars - ref_vars)

print(f"Common variables: {len(common)}")
print(f"Only in reference: {len(only_ref)}")
print(f"Only in hydro-param: {len(only_hp)}")
print()
print("Common:", common)
print()
print("Only in reference:", only_ref)
print()
print("Only in hydro-param:", only_hp[:30], "..." if len(only_hp) > 30 else "")

## Summary Statistics

For all common variables: mean, std, min, max for both datasets, plus mean absolute
difference and Pearson correlation.

In [ ]:
def compute_stats(ref_ds, hp_ds, variables):
    """Compute comparison statistics for a list of common variables.

    Aligns by position (not coordinate value) since ref uses nhm_id
    and hp uses nhru with a constant +1 offset.
    """
    rows = []
    for var in variables:
        try:
            r = ref_ds[var].values.astype(float).ravel()
            h = hp_ds[var].values.astype(float).ravel()
        except (ValueError, TypeError):
            continue
        if r.shape != h.shape:
            continue
        mask = np.isfinite(r) & np.isfinite(h)
        if mask.sum() < 2:
            continue
        r, h = r[mask], h[mask]
        corr = np.corrcoef(r, h)[0, 1] if np.std(r) > 0 and np.std(h) > 0 else np.nan
        rows.append({
            "variable": var,
            "ref_mean": np.mean(r),
            "hp_mean": np.mean(h),
            "ref_std": np.std(r),
            "hp_std": np.std(h),
            "ref_min": np.min(r),
            "hp_min": np.min(h),
            "ref_max": np.max(r),
            "hp_max": np.max(h),
            "rmse": np.sqrt(np.mean((r - h) ** 2)),
            "bias": np.mean(h - r),
            "corr": corr,
        })
    return pd.DataFrame(rows).set_index("variable").round(4)


# Separate HRU and segment variables
hru_vars = [v for v in common if ref_ds[v].dims == ("nhru",)]
seg_vars = [v for v in common if ref_ds[v].dims == ("nsegment",)]
other_vars = [v for v in common if v not in hru_vars and v not in seg_vars]

print(f"HRU variables ({len(hru_vars)}): {hru_vars}")
print(f"Segment variables ({len(seg_vars)}): {seg_vars}")
if other_vars:
    print(f"Other ({len(other_vars)}): {other_vars}")

stats_df = compute_stats(ref_ds, hp_ds, common)
print(f"\n{len(stats_df)} comparable variables")
stats_df.sort_values("corr", ascending=True)

## HRU Parameters: 1:1 Scatter Plots

Scatter plots for all HRU-dimensioned parameters, sorted by correlation (worst first).
Parameters with zero variance in either dataset are collected separately.

In [ ]:
def scatter_grid(ref_ds, hp_ds, stats_df, variables, title_prefix=""):
    """Plot 1:1 scatter grid for a set of variables, sorted by correlation."""
    # Split into plottable (spatial variation) vs uniform
    plottable, uniform = [], []
    for var in sorted(variables):
        if var not in stats_df.index:
            continue
        row = stats_df.loc[var]
        if row["ref_std"] < 1e-12 or row["hp_std"] < 1e-12:
            uniform.append(var)
        else:
            plottable.append(var)

    plottable = sorted(plottable, key=lambda v: stats_df.loc[v, "corr"])
    print(f"{len(plottable)} parameters with spatial variation, "
          f"{len(uniform)} uniform parameters")

    if not plottable:
        return uniform

    ncols = min(6, len(plottable))
    nrows = (len(plottable) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    axes = np.atleast_2d(axes)

    for idx, var in enumerate(plottable):
        ax = axes[idx // ncols, idx % ncols]
        r = ref_ds[var].values.astype(float).ravel()
        h = hp_ds[var].values.astype(float).ravel()
        mask = np.isfinite(r) & np.isfinite(h)
        r, h = r[mask], h[mask]
        ax.scatter(r, h, s=2, alpha=0.3)
        lo, hi = min(r.min(), h.min()), max(r.max(), h.max())
        margin = (hi - lo) * 0.05 if hi > lo else 1.0
        ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin], "r--", lw=0.8)
        corr = stats_df.loc[var, "corr"]
        color = "green" if corr > 0.9 else ("orange" if corr > 0.5 else "red")
        ax.set_title(f"{var}\nr={corr:.3f}", fontsize=8, color=color)
        ax.tick_params(labelsize=6)

    for idx in range(len(plottable), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.suptitle(
        f"{title_prefix}hydro-param vs Reference (sorted by r, worst first)",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    plt.show()
    return uniform


hru_uniform = scatter_grid(ref_ds, hp_ds, stats_df, hru_vars, title_prefix="HRU: ")

## Segment Parameters: 1:1 Scatter Plots

Segment-level parameters from NHD spatial join. Known issues:
- `seg_cum_area` and `seg_slope` have ~28% mismatches from spatial-join errors
- `segment_type` missing lake classification (all channels in our output)
- `seg_depth` is constant 1.0 (no empirical derivation yet)

In [ ]:
seg_uniform = scatter_grid(ref_ds, hp_ds, stats_df, seg_vars, title_prefix="Segment: ")

## Segment Diagnostic: Spatial Join Quality

For `seg_cum_area` and `seg_slope`, quantify what fraction of segments are
well-matched vs mismatched due to NHD spatial-join errors.

In [ ]:
def sjoin_quality_report(ref_ds, hp_ds, var, thresholds=(0.1, 0.5, 2.0)):
    """Report spatial-join match quality for a segment variable."""
    r = ref_ds[var].values.astype(float)
    h = hp_ds[var].values.astype(float)
    valid = (np.isfinite(r) & np.isfinite(h) & (r != 0))
    r_v, h_v = r[valid], h[valid]
    ratios = h_v / r_v

    print(f"=== {var} ({valid.sum()}/{len(r)} valid segments) ===")
    for pct in thresholds:
        n_within = np.sum((ratios > (1 - pct)) & (ratios < (1 + pct)))
        print(f"  Within {pct:.0%}: {n_within}/{len(ratios)} "
              f"({n_within / len(ratios):.0%})")

    n_way_off = np.sum((ratios > 2) | (ratios < 0.5))
    print(f"  Way off (>2x or <0.5x): {n_way_off}/{len(ratios)} "
          f"({n_way_off / len(ratios):.0%})")
    print(f"  Correlation (all): {np.corrcoef(r_v, h_v)[0, 1]:.4f}")

    # Correlation excluding outliers
    inlier = (ratios > 0.5) & (ratios < 2.0)
    if inlier.sum() > 2:
        print(f"  Correlation (inliers): "
              f"{np.corrcoef(r_v[inlier], h_v[inlier])[0, 1]:.4f}")
    print()


for var in ["seg_cum_area", "seg_slope"]:
    if var in ref_ds and var in hp_ds:
        sjoin_quality_report(ref_ds, hp_ds, var)

# segment_type distribution
if "segment_type" in ref_ds and "segment_type" in hp_ds:
    r_st = ref_ds["segment_type"].values
    h_st = hp_ds["segment_type"].values
    print("=== segment_type ===")
    print(f"  Ref unique:  {dict(zip(*np.unique(r_st, return_counts=True)))}")
    print(f"  Ours unique: {dict(zip(*np.unique(h_st, return_counts=True)))}")
    print(f"  Match: {(r_st == h_st).sum()}/{len(r_st)}")
    print(f"  Note: ref has lake (1) and type-8 segments; ours defaults all to channel (0)")


## Spatial Difference Maps: Worst Parameters

Map the spatial pattern of (hydro-param - reference) for the 12 parameters with
the lowest correlation, to identify geographic patterns in the discrepancies.

In [ ]:
# Load HRU fabric for spatial maps
gdf = gpd.read_file(FABRIC_PATH)
print(f"Loaded fabric: {len(gdf)} HRUs, CRS: {gdf.crs}")

In [ ]:
# Build list of HRU-dimensioned variables with spatial variation, sorted by correlation
spatial_plottable = [
    v for v in hru_vars
    if v in stats_df.index
    and stats_df.loc[v, "ref_std"] > 1e-12
    and stats_df.loc[v, "hp_std"] > 1e-12
]
spatial_plottable = sorted(spatial_plottable, key=lambda v: stats_df.loc[v, "corr"])

# Take worst 12
spatial_vars = spatial_plottable[:12]

ncols = 4
nrows = (len(spatial_vars) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.atleast_2d(axes)

for idx, var in enumerate(spatial_vars):
    ax = axes[idx // ncols, idx % ncols]
    r = ref_ds[var].values.astype(float).ravel()
    h = hp_ds[var].values.astype(float).ravel()
    diff = h - r
    plot_gdf = gdf.copy()
    plot_gdf["diff"] = diff
    vmax = np.nanpercentile(np.abs(diff[np.isfinite(diff)]), 95)
    if vmax < 1e-10:
        vmax = 1.0
    plot_gdf.plot(
        column="diff",
        ax=ax,
        legend=True,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        legend_kwds={"shrink": 0.6},
    )
    corr = stats_df.loc[var, "corr"]
    ax.set_title(f"{var}\nr={corr:.3f}", fontsize=9)
    ax.set_axis_off()

for idx in range(len(spatial_vars), nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

fig.suptitle("Spatial Difference Maps (hp - ref): Worst Correlation HRU Parameters", fontsize=13)
fig.tight_layout()
plt.show()

## Monthly Parameter Comparison

Parameters with a `nmonth` dimension (12 values per HRU): `jh_coef`, `dday_intcp`,
`jh_coef_hru`, `tmax_allrain_offset`, etc.

In [ ]:
monthly_vars = [
    v
    for v in common
    if "nmonth" in ref_ds[v].dims
    and "nmonth" in hp_ds[v].dims
    and ref_ds[v].shape == hp_ds[v].shape
]
print(f"Monthly parameters to compare: {monthly_vars}")

if monthly_vars:
    ncols = min(4, len(monthly_vars))
    nrows = (len(monthly_vars) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.atleast_2d(axes)

    months = np.arange(1, 13)
    for idx, var in enumerate(monthly_vars):
        ax = axes[idx // ncols, idx % ncols]
        # Mean across HRUs for each month
        r = ref_ds[var].values.astype(float)
        h = hp_ds[var].values.astype(float)
        # Shape is (nmonth, nhru) or (nhru, nmonth) — find month axis
        month_axis = list(ref_ds[var].dims).index("nmonth")
        r_mean = np.nanmean(r, axis=1 - month_axis)  # mean over HRUs
        h_mean = np.nanmean(h, axis=1 - month_axis)
        r_std = np.nanstd(r, axis=1 - month_axis)
        h_std = np.nanstd(h, axis=1 - month_axis)

        ax.errorbar(months - 0.1, r_mean, yerr=r_std, fmt="o-", ms=4, label="Reference", capsize=2)
        ax.errorbar(
            months + 0.1, h_mean, yerr=h_std, fmt="s-", ms=4, label="hydro-param", capsize=2
        )
        ax.set_title(var)
        ax.set_xlabel("Month")
        ax.set_xticks(months)
        ax.legend(fontsize=7)

    for idx in range(len(monthly_vars), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.tight_layout()
    plt.show()

## Triage Summary

Classify all parameters by agreement quality to identify refinement priorities.

In [ ]:
def classify(row):
    """Classify parameter agreement into triage categories."""
    corr = row["corr"]

    if row["ref_std"] < 1e-12 and row["hp_std"] < 1e-12:
        return "uniform-match" if row["rmse"] < 1e-6 else "uniform-differs"
    if row["ref_std"] < 1e-12 or row["hp_std"] < 1e-12:
        return "one-uniform"
    if np.isnan(corr):
        return "no-correlation"
    if corr > 0.9:
        return "good (r>0.9)"
    if corr > 0.7:
        return "fair (r>0.7)"
    if corr > 0.3:
        return "weak (r>0.3)"
    return "poor (r<0.3)"


stats_df["category"] = stats_df.apply(classify, axis=1)

print("=== Parameter Agreement Summary ===\n")
for cat in [
    "good (r>0.9)",
    "fair (r>0.7)",
    "weak (r>0.3)",
    "poor (r<0.3)",
    "one-uniform",
    "uniform-match",
    "uniform-differs",
    "no-correlation",
]:
    members = stats_df[stats_df["category"] == cat].index.tolist()
    if members:
        print(f"{cat}: {len(members)} parameters")
        for v in members:
            row = stats_df.loc[v]
            corr_str = f"r={row['corr']:.3f}" if not np.isnan(row["corr"]) else "r=NaN"
            print(f"  {v:30s}  ref={row['ref_mean']:10.4f}  hp={row['hp_mean']:10.4f}  {corr_str}")
        print()

# Color-coded summary bar
cats = stats_df["category"].value_counts()
fig, ax = plt.subplots(figsize=(10, 1.5))
colors = {
    "good (r>0.9)": "#2ecc71",
    "fair (r>0.7)": "#f1c40f",
    "weak (r>0.3)": "#e67e22",
    "poor (r<0.3)": "#e74c3c",
    "uniform-match": "#3498db",
    "uniform-differs": "#9b59b6",
    "one-uniform": "#95a5a6",
    "no-correlation": "#bdc3c7",
}
left = 0
for cat in colors:
    if cat in cats:
        ax.barh(0, cats[cat], left=left, color=colors[cat], label=f"{cat} ({cats[cat]})")
        left += cats[cat]
ax.set_yticks([])
ax.set_xlabel("Number of parameters")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("Parameter Agreement Classification")
fig.tight_layout()
plt.show()

## Forcing Comparison

Compare forcing files if available.

In [ ]:
HP_FORCING = Path("../pw-check/models/pywatershed/forcing")
REF_FORCING = REF_PARAM.parent

# Check what forcing files exist
hp_forcing_files = sorted(HP_FORCING.glob("*.nc")) if HP_FORCING.exists() else []
ref_forcing_files = sorted(REF_FORCING.glob("*.nc"))

print(f"hydro-param forcing: {[f.name for f in hp_forcing_files]}")
print(f"Reference forcing: {[f.name for f in ref_forcing_files]}")

In [ ]:
forcing_vars = ["prcp", "tmax", "tmin"]

for fvar in forcing_vars:
    hp_file = HP_FORCING / f"{fvar}.nc" if HP_FORCING.exists() else None
    ref_file = REF_FORCING / f"{fvar}.nc"
    if hp_file and hp_file.exists() and ref_file.exists():
        hp_f = xr.open_dataset(hp_file)
        ref_f = xr.open_dataset(ref_file)
        # Find overlapping time
        common_time = np.intersect1d(hp_f.time.values, ref_f.time.values)
        if len(common_time) > 0:
            # Determine HRU dimension name for each dataset
            hp_hru_dim = "nhru" if "nhru" in hp_f[fvar].dims else "nhm_id"
            ref_hru_dim = "nhru" if "nhru" in ref_f[fvar].dims else "nhm_id"
            hp_sel = hp_f[fvar].sel(time=common_time).mean(dim=hp_hru_dim)
            ref_sel = ref_f[fvar].sel(time=common_time).mean(dim=ref_hru_dim)
            fig, ax = plt.subplots(figsize=(10, 3))
            ref_sel.plot(ax=ax, label="Reference", alpha=0.7)
            hp_sel.plot(ax=ax, label="hydro-param", alpha=0.7)
            ax.set_title(f"{fvar}: domain-mean time series")
            ax.legend()
            plt.tight_layout()
            plt.show()
        else:
            print(f"{fvar}: no overlapping time steps")
        hp_f.close()
        ref_f.close()
    else:
        print(f"{fvar}: files not found in both locations")